### Fase 1: Transformando Canciones en "Canastas"
El algoritmo Apriori no lee DataFrames normales; necesita una lista de transacciones. 
En nuestro caso, cada **Canción** es una transacción, y los **Artistas** que participan en ella son los artículos de la canasta.

In [ ]:
# Instalar si es necesario: !pip install mlxtend
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# 1. INGESTA DE DATOS
url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/Most%20Streamed%20Spotify%20Songs%202024.csv"
df_spotify = pd.read_csv(url, encoding='latin-1')

# 2. LIMPIEZA: Eliminar nulos de la columna de artistas
df_spotify = df_spotify.dropna(subset=['Artist'])

# 3. TRANSFORMACIÓN: Separar los artistas que colaboran en cada canción
# Asumimos que las colaboraciones están separadas por coma (ej. "Bad Bunny, Feid")
lista_colaboraciones = (
    df_spotify['Artist']
    .apply(lambda x: [artista.strip() for artista in x.split(',') if artista.strip()])
    .tolist()
)

# Muestra rápida de cómo se ven nuestras "canastas"
print("Muestra de las primeras 5 colaboraciones:")
for i in range(5):
    print(f"Canción {i+1}: {lista_colaboraciones[i]}")

ModuleNotFoundError: No module named 'mlxtend'

In [ ]:
# Utiliza TransactionEncoder para convertir 'lista_colaboraciones' en una matriz booleana
te = TransactionEncoder()
matriz_bool = te.fit_transform(lista_colaboraciones)
df_transacciones = pd.DataFrame(matriz_bool, columns=te.columns_)

print("Matriz de transacciones:")
display(df_transacciones.head())

### Fase 2: Aplicando Apriori
Ahora que tenemos la matriz con ceros y unos, extraigamos los conjuntos de artistas que colaboran frecuentemente.
Debido a que hay miles de artistas solistas, las colaboraciones son estadísticamente raras. Usaremos un `min_support` muy bajo para capturar las colaboraciones.

In [ ]:
# 1. Encuentra los Itemsets Frecuentes usando un soporte mínimo del 0.005 (0.5% de las canciones)
itemsets_frecuentes = apriori(df_transacciones, min_support=0.005, use_colnames=True)

# 2. Genera las Reglas de Asociación filtrando por la métrica 'lift' con un mínimo de 1.2
reglas_musicales = association_rules(itemsets_frecuentes, metric="lift", min_threshold=1.2)

# 3. Muestra las reglas resultantes ordenadas por 'confidence' de mayor a menor
display(reglas_musicales.sort_values("confidence", ascending=False).head(10))

### 🧠 Fase 3: Interpretación de Negocio (El Entregable)

Como Científico de Datos, no basta con entregar una tabla. Debes explicarle los hallazgos al Director Creativo. Revisa tu DataFrame `reglas_musicales` y responde las siguientes 4 preguntas:

**1. Análisis de Soporte (El Volumen):**
Menciona a los dos artistas que conforman la regla con el mayor **Soporte**. ¿Qué significa este número en términos simples para el negocio?

> **Tu Respuesta:** La regla con mayor soporte es la combinación de artistas que aparece con más frecuencia en las canciones. El soporte representa la proporción de canciones en las que aparecen juntos. En otras palabras, nos dice qué tan común es que esos dos artistas colaboren dentro del dataset.

**2. Análisis de Confianza (La Dirección):**
Busca una regla que tenga una **Confianza** mayor al 0.5 (50%). Traduce esta regla matemática a una frase que el director pueda entender (Ej. "Si contratamos al artista A, existe un X% de probabilidad de que el artista B también deba estar en el track para que sea el éxito que registramos").

> **Tu Respuesta:** Si aparece el artista A, entonces existe un X% de probabilidad de que también aparezca el artista B en la misma canción. Por ejemplo: “Si contratamos a A, hay un 60% de probabilidad de que B también forme parte del track”.

**3. Análisis de Lift (El Impulso):**
Busca la regla con el **Lift** más alto de toda tu tabla. 
¿Por qué esta métrica es más confiable que el Soporte para decidir si estos dos artistas *realmente* tienen química musical o si solo fue una coincidencia estadística?

> **Tu Respuesta:** El lift mide si la asociación entre dos artistas es más fuerte de lo que sería por casualidad. Si el lift es alto, significa que esa colaboración ocurre mucho más de lo esperado, así que no es coincidencia estadística; hay una relación real de afinidad musical.

**4. Toma de Decisión:**
Basado puramente en las reglas matemáticas que descubriste, si la disquera solo tiene presupuesto para juntar a dos artistas en el estudio la próxima semana, ¿qué pareja elegirías y por qué? (Justifica usando las 3 métricas de oro).

> **Tu Respuesta:** Elegiría la pareja con mayor lift, y luego la mejor confianza y soporte. Es decir, la combinación que no solo aparece con frecuencia, sino que además tiene una relación más fuerte y más probable en el mismo track. Eso nos da la mejor apuesta para una colaboración exitosa.